In [4]:
import os
import shutil
from pathlib import Path

# Possible original dirs
orig_img_dirs = [
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\test",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\train",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\images\val",
]
orig_label_dirs = [
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\test",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\train",
    r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\labels\val",
]

adv_test = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\images"

dest_img = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test\images"
dest_labels = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test\labels"

# Ensure destination directories exist
os.makedirs(dest_img, exist_ok=True)
os.makedirs(dest_labels, exist_ok=True)

# Collect adv_test image names
adv_images = [Path(f).stem for f in os.listdir(adv_test) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

copied = 0
missing = []

for name in adv_images:
    found = False
    for img_dir, lbl_dir in zip(orig_img_dirs, orig_label_dirs):
        src_img_jpg = Path(img_dir) / f"{name}.jpg"
        src_img_png = Path(img_dir) / f"{name}.png"
        src_label = Path(lbl_dir) / f"{name}.txt"

        if src_img_jpg.exists() or src_img_png.exists():
            src_img = src_img_jpg if src_img_jpg.exists() else src_img_png

            # Copy image
            shutil.copy2(src_img, Path(dest_img) / src_img.name)

            # Copy label if exists
            if src_label.exists():
                shutil.copy2(src_label, Path(dest_labels) / src_label.name)
            else:
                print(f"⚠️ No label for {name}")

            copied += 1
            found = True
            break  # stop searching once found in one split

    if not found:
        missing.append(name)

print(f"✅ Done! Copied {copied} images.")
if missing:
    print("❌ Missing originals for:", missing)


✅ Done! Copied 204 images.


## Load necessary library

In [2]:
import os
from ultralytics import YOLO
from pathlib import Path

In [2]:
model = YOLO("yolov5s.pt")

PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 17.7M/17.7M [00:00<00:00, 21.7MB/s]


## Original vs Adversarial mAP50

In [8]:
img_dir_clean = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test"
img_dir_adv = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test"

results = model.val(data=os.path.join(img_dir_adv, "data.yaml"), split="test", imgsz=640)
map50_per_class = results.box.maps  

# class 0 = person
map50_person = map50_per_class[0]

print("mAP@0.5 for person:", map50_person)


Ultralytics 8.3.167  Python-3.13.3 torch-2.6.0+cpu CPU (13th Gen Intel Core(TM) i5-13420H)
val: Fast image access  (ping: 0.40.4 ms, read: 133.7121.2 MB/s, size: 370.7 KB)


val: Scanning C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\labels... 204 images, 0 backgrounds, 0 corrupt: 100%|

val: New cache created: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:20<00:00,  1.55s/it]


                   all        204       1260      0.291      0.265      0.254      0.121
                person        204       1013      0.583       0.53      0.504      0.241
               bicycle        204        247          0          0    0.00312    0.00217
Speed: 2.3ms preprocess, 75.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to C:\Adrianov\Projects\Project-Satanael\runs\detect\val3
mAP@0.5 for person: 0.24076781114927495


## Attack Success Rate

In [12]:
import os, glob
import numpy as np
from ultralytics import YOLO

# --- CONFIG ---
IMG_DIR_CLEAN = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\clean_test\images"
IMG_DIR_ADV   = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\images"
PERSON_CLASS_ID = 0           # class 0 = person
CONF_THR = 0.25               # prediction confidence threshold
IOU_THR  = 0.50               # IoU threshold to count a GT as detected
WEIGHTS  = "yolov5s.pt"       # pretrained model
DEVICE   = None               # e.g. "cuda:0" or None for auto

# --- HELPERS ---
def replace_last_segment(path, old="images", new="labels"):
    parts = os.path.normpath(path).split(os.sep)
    for i in range(len(parts)-1, -1, -1):
        if parts[i].lower() == old:
            parts[i] = new
            return os.sep.join(parts)
    return path  # fallback: unchanged

def guess_label_path(image_path):
    stem = os.path.splitext(os.path.basename(image_path))[0]
    # try .../labels/.../stem.txt (YOLO standard)
    p1 = os.path.join(os.path.dirname(replace_last_segment(image_path, "images", "labels")), f"{stem}.txt")
    if os.path.isfile(p1): return p1
    # try same directory
    p2 = os.path.join(os.path.dirname(image_path), f"{stem}.txt")
    if os.path.isfile(p2): return p2
    return None

def xywhn_to_xyxy_px(xywhn, img_w, img_h):
    # xywhn: [xc, yc, w, h] normalized; return [x1,y1,x2,y2] in pixels
    xc, yc, w, h = xywhn
    xc *= img_w; yc *= img_h; w *= img_w; h *= img_h
    x1 = xc - w/2.0; y1 = yc - h/2.0; x2 = xc + w/2.0; y2 = yc + h/2.0
    return np.array([x1, y1, x2, y2], dtype=np.float32)

def iou_matrix(boxes_a, boxes_b):
    # boxes: Nx4 or Mx4 in xyxy
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    A = boxes_a[:, None, :]  # N,1,4
    B = boxes_b[None, :, :]  # 1,M,4
    inter_x1 = np.maximum(A[...,0], B[...,0])
    inter_y1 = np.maximum(A[...,1], B[...,1])
    inter_x2 = np.minimum(A[...,2], B[...,2])
    inter_y2 = np.minimum(A[...,3], B[...,3])
    inter_w = np.clip(inter_x2 - inter_x1, 0, None)
    inter_h = np.clip(inter_y2 - inter_y1, 0, None)
    inter = inter_w * inter_h
    area_a = (A[...,2]-A[...,0]) * (A[...,3]-A[...,1])
    area_b = (B[...,2]-B[...,0]) * (B[...,3]-B[...,1])
    union = area_a + area_b - inter + 1e-9
    return (inter / union).astype(np.float32)

def greedy_match(gt_boxes, pred_boxes, iou_thr):
    """
    Returns: matched_gt_indices (set), used_pred_indices (set)
    Each GT is matched to at most one prediction via highest IoU >= iou_thr.
    """
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return set(), set()
    ious = iou_matrix(gt_boxes, pred_boxes)
    matches = []
    # collect all candidate pairs above threshold
    gt_idx, pred_idx = np.where(ious >= iou_thr)
    for g, p in zip(gt_idx, pred_idx):
        matches.append((g, p, ious[g, p]))
    # sort by IoU desc and do greedy selection
    matches.sort(key=lambda x: x[2], reverse=True)
    used_g, used_p = set(), set()
    for g, p, _ in matches:
        if g not in used_g and p not in used_p:
            used_g.add(g); used_p.add(p)
    return used_g, used_p

# --- LOAD MODEL & RUN INFERENCE ---
model = YOLO(WEIGHTS)

# run prediction on both dirs; map results by basename to avoid ordering issues
def run_dir(dir_path):
    res = model.predict(source=dir_path, imgsz=640, conf=CONF_THR, device=DEVICE, stream=False, verbose=False)
    by_name = {}
    for r in res:
        name = os.path.basename(r.path)
        by_name[name] = r
    return by_name

clean_results = run_dir(IMG_DIR_CLEAN)
adv_results   = run_dir(IMG_DIR_ADV)

# collect common image names
clean_names = set(clean_results.keys())
adv_names   = set(adv_results.keys())
common = sorted(clean_names & adv_names)

if not common:
    raise RuntimeError("No matching filenames between clean and adversarial directories.")

# --- CORE EVALUATION ---
tp_clean_total = 0  # number of GT person objects detected on clean (credit these only)
lost_in_adv    = 0  # among those, how many are missed on adversarial images

skipped_no_label = 0

for name in common:
    r_clean = clean_results[name]
    r_adv   = adv_results[name]

    # Get image size from clean result (H,W)
    Hc, Wc = r_clean.orig_shape
    Ha, Wa = r_adv.orig_shape

    # --- load GT labels for this image (YOLO format) ---
    label_path = guess_label_path(r_clean.path)
    if label_path is None or not os.path.isfile(label_path):
        skipped_no_label += 1
        continue

    gt_person_xyxy = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: 
                continue
            cls = int(float(parts[0]))
            if cls != PERSON_CLASS_ID:
                continue
            xc, yc, w, h = map(float, parts[1:5])
            gt_person_xyxy.append(xywhn_to_xyxy_px([xc, yc, w, h], Wc, Hc))
    gt_person_xyxy = np.array(gt_person_xyxy, dtype=np.float32)
    if gt_person_xyxy.size == 0:
        continue  # no GT persons in this image

    # --- predictions (person class only) ---
    def person_preds_xyxy(result):
        if result.boxes is None or len(result.boxes) == 0:
            return np.empty((0,4), dtype=np.float32)
        cls = result.boxes.cls.cpu().numpy()
        conf = result.boxes.conf.cpu().numpy()
        xyxy = result.boxes.xyxy.cpu().numpy()
        m = (cls == PERSON_CLASS_ID) & (conf >= CONF_THR)
        return xyxy[m].astype(np.float32)

    preds_clean = person_preds_xyxy(r_clean)
    preds_adv   = person_preds_xyxy(r_adv)

    # --- match GT->clean preds to find which GT persons are truly detected on clean ---
    matched_g_clean, used_p_clean = greedy_match(gt_person_xyxy, preds_clean, IOU_THR)
    tp_clean_total += len(matched_g_clean)
    if len(matched_g_clean) == 0:
        continue

    # If adv resolution differs (shouldn't), scale GT to adv space
    if (Hc, Wc) != (Ha, Wa):
        scale_x = Wa / float(Wc)
        scale_y = Ha / float(Hc)
        gt_for_adv = gt_person_xyxy.copy()
        gt_for_adv[:, [0,2]] *= scale_x
        gt_for_adv[:, [1,3]] *= scale_y
    else:
        gt_for_adv = gt_person_xyxy

    # consider only those GTs that were TP on clean
    gt_tp_clean = gt_for_adv[list(matched_g_clean), :]

    # --- check if those same GTs are detected on adversarial ---
    # match GT_tp_clean to adv preds
    matched_g_adv, used_p_adv = greedy_match(gt_tp_clean, preds_adv, IOU_THR)
    missed_here = len(gt_tp_clean) - len(matched_g_adv)
    lost_in_adv += missed_here

# --- FINAL METRIC ---
if tp_clean_total == 0:
    print("No GT person objects were detected on clean images (tp_clean_total=0). Cannot compute ASR.")
else:
    asr = lost_in_adv / tp_clean_total
    print(f"GT persons detected on clean (denominator): {tp_clean_total}")
    print(f"Of those, missed on adversarial:           {lost_in_adv}")
    print(f"Attack Success Rate (ASR):                 {asr:.2%}")

if skipped_no_label > 0:
    print(f"Note: skipped {skipped_no_label} images with missing label files.")


PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

GT persons detected on clean (denominator): 646
Of those, missed on adversarial:           100
Attack Success Rate (ASR):                 15.48%


## End-to-End PAD

In [2]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2
import sys
import glob
import os
import importlib.util
from segment_anything import SamPredictor, SamAutomaticMaskGenerator, sam_model_registry

import math
from PIL import Image

In [3]:
fusefilter_path = os.path.abspath("../defenselib/fuse_filter.py")

spec = importlib.util.spec_from_file_location("fuse_filter", fusefilter_path)
fusefilter = importlib.util.module_from_spec(spec)
sys.modules["fusefilter"] = fusefilter
spec.loader.exec_module(fusefilter)


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="The number of unique classes is greater than 50%",
    category=UserWarning
)


In [5]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [ ]:
from fusefilter import fuse_heatmap, heatmap_filter

iou_thre = 0.5
ratio_mi = 0.5 # ratio_cd = 1-ratio_mi
kernel_pram = 80
thresh_pram = 80 # percentile, from small to big
input_path = r"C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\patched_random_bigger_test\images"
save_path = r"C:\Adrianov\Projects\Project-Satanael\pad_res"

def get_mask(image, mask_generator):
    
    masks = mask_generator.generate(image.astype(np.uint8))
    return masks

if __name__ == "__main__":
    device = "cuda:0"
    # sam = sam_model_registry["vit_b"](checkpoint="models/sam_vit_b_01ec64.pth")
    torch.cuda.empty_cache()
    sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
    sam.to(device=device)
    mask_generator = SamAutomaticMaskGenerator(sam)

    print(save_path)
    folder = os.path.exists(save_path)

    if not folder:
        os.makedirs(save_path)

    with torch.no_grad():
        data_dir = input_path
        data_files = os.listdir(data_dir)
        i = len(processed_files)
        start = time.time()
        # processed_files = []
        for data_file in data_files:
            print(data_file)
            if data_file in processed_files:
                continue
            name = data_file.split(".")[0]
            impath = os.path.join(data_dir, data_file)
            
            ori_img = Image.open(impath).convert('RGB')
            ori_width, ori_height = ori_img.size
            print("ori_height , ori_width", ori_height, ori_width)

            mi_img, cd_img, fuse_img = fuse_heatmap(impath, ori_height, ori_width)

            threshold = np.percentile(fuse_img, thresh_pram)
            h_t, h_t_o, h_t_o_c, h_t_o_c_o = heatmap_filter(fuse_img, threshold, ori_height, ori_width)

            gray = np.where(h_t_o_c_o >0,1,0)

            rgb_color = cv2.imread(impath)

            image = cv2.cvtColor(rgb_color, cv2.COLOR_BGR2RGB)
            
            #just for Dpatch
            #image = cv2.resize(image,(416,416))

            h = image.shape[0]
            w = image.shape[1]
            mask = get_mask(image, mask_generator)

            result_mask = np.zeros((h,w))
            for k in range(len(mask)):

                mask_k = mask[k].get('segmentation')
                n = mask_k&gray
                u = mask_k #|gray
                iou = np.sum(n)/(np.sum(u))
                print("iou",iou)

                n_1 = mask_k&result_mask.astype(np.uint8)
                u_1 = mask_k
                iou1 =  np.sum(n_1)/(np.sum(u_1))
                print("iou1",iou1)

                if(iou>iou_thre and iou1<0.1):
                    mask_k_save = np.expand_dims(mask_k,axis=2)
                    mask_k_save = np.tile(mask_k_save,3)
                    rgb_color = rgb_color*(~mask_k_save) 
                    result_mask = result_mask.astype(np.uint8) | mask_k
                    '''mask_k_save = np.expand_dims(mask_k,axis=2)
                    mask_k_save = np.tile(mask_k_save,3)
                    mask_gray = np.expand_dims(mask_k*128,axis=2)
                    mask_gray = np.tile(mask_gray,3)
                    rgb_color = rgb_color*(~mask_k_save) + mask_gray
                    result_mask = result_mask.astype(np.uint8) | mask_k'''
                    '''result_mask = result_mask.astype(np.uint8) | mask_k
                    rgb_color = cv2.inpaint(rgb_color, mask_k.astype(np.uint8), 3, cv2.INPAINT_NS)'''

            
            save_heatmap(h_t_o_c_o, os.path.join(save_path, name + "_h_t_o_c_o.png"))
            save_heatmap(result_mask, os.path.join(save_path, name + "_pad_mask.png"))
            # cv2.imwrite(os.path.join(save_path, name+".png"),rgb_color)
            i+=1
            elapsed = time.time() - start
            print(f"{i} images processed. {elapsed+1396.69:.2f} seconds elasped. Avg inference time: {(elapsed+1396.69)/i:.2f}")
            processed_files.append(data_file)

C:\Adrianov\Projects\Project-Satanael\.venv311\Lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


C:\Adrianov\Projects\Project-Satanael\pad_res
1499563559079.jpg
1499563792315.jpg
1499563803890.jpg
1499563861719.jpg
1499564901648.jpg
1499564959353.jpg
1499565215708.jpg
1499565240325.jpg
1499565299106.jpg
1499568775385.jpg
1499569799136.jpg
1499581341885.jpg
ori_height , ori_width 1200 1624


In [21]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())


None
False


In [26]:
!nvidia-smi


Sun Aug 31 22:40:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.90                 Driver Version: 565.90         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   38C    P8              1W /   75W |     125MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
import gc, torch

del sam        # or your model variable
del mask_generator
torch.cuda.empty_cache()
gc.collect()


2525

In [27]:
import os
import re
results_dir = r"C:\Adrianov\Projects\Project-Satanael\pad_res"

# Get all file names in the directory
files = os.listdir(results_dir)

unique_bases = set()

for f in files:
    match = re.match(r"(.+?_pad_mask\.png)$", f)
    if match:
        unique_bases.add(match.group(1))
        
processed_files = list(unique_bases)
processed_files = [f.replace("_pad_mask.png", ".jpg") for f in processed_files]
print(processed_files)
print(len(processed_files))

['1499563559079.jpg', '1499569799136.jpg', '1499563792315.jpg', '1499563803890.jpg', '1499565299106.jpg', '1499568775385.jpg', '1499564901648.jpg', '1499565240325.jpg', '1499565215708.jpg', '1499564959353.jpg', '1499563861719.jpg']
11


In [28]:
processed_files

['1499563559079.jpg',
 '1499569799136.jpg',
 '1499563792315.jpg',
 '1499563803890.jpg',
 '1499565299106.jpg',
 '1499568775385.jpg',
 '1499564901648.jpg',
 '1499565240325.jpg',
 '1499565215708.jpg',
 '1499564959353.jpg',
 '1499563861719.jpg']